# VaakMitra: IndicConformer phoneme transfer + edge-student distillation

This notebook executes the two GPU tracks required by Member 2:

1. load the pinned gated Tamil IndicConformer, attach a new phoneme CTC head, train head-only,
   unfreeze the last four encoder blocks, and optionally unfreeze the full encoder;
2. extract the selected reference encoder's representations, train compact Conformer and
   Conv-BiGRU students, evaluate adult-proxy PER, export ONNX/INT8 candidates, and write a
   digest-bound results bundle to Google Drive.

**Runtime:** use a paid Colab GPU when possible. A T4 can use batch size 1 with accumulation;
24 GB or more is recommended. Full-corpus work spans multiple sessions, so every expensive phase
resumes from Google Drive.

**Evidence boundary:** OpenSLR 127 is adult Tamil engineering-proxy evidence. This notebook never
claims child/ASD accuracy, therapist calibration, tablet performance, clinical readiness, or
production readiness. If the clinician has not completed the inventory review, the notebook can
run only as an explicitly provisional research experiment; those checkpoints must be retrained if
the inventory changes.

Before running, accept the gated model conditions on Hugging Face and add `HF_TOKEN` to Colab
Secrets. Never paste the token into this notebook.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

# User controls. Keep RUN_MODE="full" for real outputs; "smoke" validates the pipeline cheaply.
RUN_MODE = "full"  # "full" or "smoke"
ALLOW_PROVISIONAL_RESEARCH_RUN = True
DOWNLOAD_CORPUS_IF_MISSING = True
FORCE_REBUILD_TARGETS = False

# Full-mode schedules. Resume checkpoints make it safe to run one or two epochs per Colab session.
REFERENCE_EPOCHS = {"head_only": 3, "top_encoder_blocks": 3, "full_encoder": 2}
STUDENT_EPOCHS = 5
BATCH_SIZE = 1
STUDENT_BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 8
NUM_WORKERS = 2
MAX_AUDIO_SECONDS = 30.0
FEATURE_SHARD_SIZE = 128
SEED = 17

# Smoke mode never counts as real evaluation evidence.
if RUN_MODE == "smoke":
    REFERENCE_EPOCHS = {"head_only": 1, "top_encoder_blocks": 1, "full_encoder": 1}
    STUDENT_EPOCHS = 1
    GRADIENT_ACCUMULATION = 1
    FEATURE_SHARD_SIZE = 16
    SPLIT_LIMITS = {"train": 64, "validation": 16, "test": 16}
elif RUN_MODE == "full":
    SPLIT_LIMITS = {"train": None, "validation": None, "test": None}
else:
    raise ValueError("RUN_MODE must be 'full' or 'smoke'")

TEACHER_MODEL_ID = "ai4bharat/indicconformer_stt_ta_hybrid_ctc_rnnt_large"
TEACHER_REVISION = "8c31aa8d04964b8fc87e4eaaee7916f7d2c024da"
AI4BHARAT_NEMO_REVISION = "8dce88cf8e94963e2033c3137f7b9993b51db88a"
ARCHIVE_URL = "https://openslr.trmal.net/resources/127/mile_tamil_asr_corpus.tar.gz"
ARCHIVE_SIZE = 13_803_410_250
ARCHIVE_SHA256 = "0ec67ad6fc6b5f48aa0f2668722dbdb2642b66c4b49c289c2849ee603f35c8c2"

EXPECTED_PRIVATE_HASHES = {
    "records.jsonl": "eaf9ffd9ebae82eb6b3fad402e15073e34136b10392f1c78275bb1fa35ccbdab",
    "speaker-assignments.json": "47c7b94d261ac2b4175049fc2631a35ab0dc0a89f02a8d46fee5188688421804",
    "full-lexicon.json": "6b0989f9c3fcfcd20d2691d0ba0dd33e070b71dc330c2115dec6fb309c2dc44d",
    "tamil-phoneme-candidate.json": "2fcd1b16afe0ffa423028f6cdce461102258ca548f07895b714b864668a07a47",
    "openslr127-frozen-index.json": "4fe2602d31d1c48ec0df775cb2011062a720559afd7d48f9678143debbb680c3",
    "openslr127-corpus-evidence.json": "78758c6c67d2b8b3e95002164863dede79e9ee234ec2e1b591af5070351f39a2",
}

DRIVE_ROOT = Path("/content/drive/MyDrive/VaakMitraGPU")
DRIVE_INPUTS = DRIVE_ROOT / "inputs"
DRIVE_RUN = DRIVE_ROOT / "runs" / f"indicconformer-{RUN_MODE}-v1"
DRIVE_PRIVATE = DRIVE_RUN / "private"
DRIVE_OUTPUTS = DRIVE_RUN / "outputs"
WORK_ROOT = Path("/content/vaakmitra-gpu-work")
ARCHIVE_PATH = WORK_ROOT / "mile_tamil_asr_corpus.tar.gz"
EXTRACT_ROOT = WORK_ROOT / "extracted"
METADATA_ROOT = WORK_ROOT / "private-metadata"

print({"run_mode": RUN_MODE, "drive_run": str(DRIVE_RUN), "provisional_allowed": ALLOW_PROVISIONAL_RESEARCH_RUN})

In [ ]:
import subprocess
import sys


def run(command: list[str], *, cwd: Path | None = None) -> None:
    print("+", " ".join(command))
    subprocess.run(command, cwd=cwd, check=True)


# CondaColab installation is state-driven and requests exactly one runtime restart.
# After that expected restart, rerun the configuration cell and this cell.
conda_ready = False
try:
    import condacolab

    condacolab.check()
    conda_ready = True
except (ImportError, AssertionError):
    run([sys.executable, "-m", "pip", "install", "-q", "condacolab==0.1.10"])
    import condacolab

    print("Installing CondaLab and restarting the runtime once.")
    condacolab.install()

if conda_ready:
    run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--extra-index-url",
            "https://download.pytorch.org/whl/cu121",
            "torch==2.2.0+cu121",
            "torchaudio==2.2.0+cu121",
            "numpy==1.26.4",
            "huggingface_hub==0.23.2",
            "pydantic>=2.8,<3",
            "soundfile>=0.12,<1",
            "onnx>=1.16,<2",
            "onnxruntime>=1.18,<2",
            "tqdm>=4.66,<5",
        ]
    )

    nemo_source = Path("/content/ai4bharat-nemo")
    if not nemo_source.exists():
        run(["git", "clone", "--filter=blob:none", "https://github.com/AI4Bharat/NeMo.git", str(nemo_source)])
    run(["git", "fetch", "origin", AI4BHARAT_NEMO_REVISION, "--depth", "1"], cwd=nemo_source)
    run(["git", "checkout", "--detach", AI4BHARAT_NEMO_REVISION], cwd=nemo_source)
    run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[asr]"], cwd=nemo_source)

    import torch

    assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU before continuing"
    print({"python": sys.version, "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0)})

In [ ]:
import shutil
import zipfile

from google.colab import drive

drive.mount("/content/drive")
for directory in (DRIVE_INPUTS, DRIVE_PRIVATE, DRIVE_OUTPUTS, WORK_ROOT, METADATA_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

metadata_zip = DRIVE_INPUTS / "vaakmitra-colab-private-metadata.zip"
if not metadata_zip.is_file():
    raise FileNotFoundError(
        f"Upload vaakmitra-colab-private-metadata.zip to {DRIVE_INPUTS}. "
        "This private bundle contains the frozen index, lexicon, and candidate—not audio."
    )

with zipfile.ZipFile(metadata_zip) as archive:
    expected_names = set(EXPECTED_PRIVATE_HASHES)
    actual_names = {Path(name).name for name in archive.namelist() if not name.endswith("/")}
    if actual_names != expected_names:
        raise ValueError(f"metadata bundle names differ: {actual_names ^ expected_names}")
    for member in archive.infolist():
        if member.is_dir():
            continue
        name = Path(member.filename)
        if name.is_absolute() or ".." in name.parts:
            raise ValueError("unsafe metadata ZIP path")
        target = METADATA_ROOT / name.name
        with archive.open(member) as source, target.open("wb") as output:
            shutil.copyfileobj(source, output, length=4 * 1024 * 1024)

print({"metadata_bundle": str(metadata_zip), "metadata_files": len(EXPECTED_PRIVATE_HASHES)})

In [ ]:
import hashlib
import json
import math
import random
import re
import tarfile
import time
import unicodedata
from collections import Counter
from dataclasses import dataclass
from pathlib import PurePosixPath
from typing import Any, Iterator, Sequence

import numpy as np
import soundfile as sf
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json_atomic(path: Path, payload: object) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.parent.mkdir(parents=True, exist_ok=True)
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    os.replace(temporary, path)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def safe_extract_tar(archive_path: Path, destination: Path) -> dict[str, int]:
    marker = destination / ".vaakmitra-extraction.json"
    if marker.is_file():
        state = json.loads(marker.read_text(encoding="utf-8"))
        if state.get("archive_sha256") == ARCHIVE_SHA256 and state.get("file_count") == 178_802:
            return state
        raise ValueError("existing extraction marker does not match the frozen archive")
    if destination.exists() and any(destination.iterdir()):
        raise ValueError("unverified extraction destination is non-empty")
    destination.mkdir(parents=True, exist_ok=True)
    resolved_root = destination.resolve()
    seen: set[str] = set()
    file_count = 0
    member_count = 0
    uncompressed_bytes = 0
    with tarfile.open(archive_path, "r:gz") as source:
        members = source.getmembers()
        for member in members:
            member_count += 1
            pure = PurePosixPath(member.name)
            if pure.is_absolute() or ".." in pure.parts or not pure.parts:
                raise ValueError(f"unsafe archive path: {member.name!r}")
            if member.issym() or member.islnk() or not (member.isfile() or member.isdir()):
                raise ValueError(f"unsafe archive member type: {member.name!r}")
            normalized = pure.as_posix().rstrip("/")
            if normalized in seen:
                raise ValueError(f"duplicate archive destination: {normalized}")
            seen.add(normalized)
            target = (resolved_root / Path(*pure.parts)).resolve()
            target.relative_to(resolved_root)
            if member.isfile():
                file_count += 1
                uncompressed_bytes += member.size
        if (member_count, file_count, uncompressed_bytes) != (178_809, 178_802, 17_314_415_289):
            raise ValueError("archive member evidence does not match the frozen corpus")
        for member in tqdm(members, desc="Safe extraction"):
            pure = PurePosixPath(member.name)
            target = resolved_root / Path(*pure.parts)
            if member.isdir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            stream = source.extractfile(member)
            if stream is None:
                raise ValueError("validated regular file has no readable stream")
            with stream, target.open("xb") as output:
                shutil.copyfileobj(stream, output, length=4 * 1024 * 1024)
    state = {
        "archive_sha256": ARCHIVE_SHA256,
        "member_count": member_count,
        "file_count": file_count,
        "uncompressed_bytes": uncompressed_bytes,
    }
    write_json_atomic(marker, state)
    return state


def verify_metadata() -> None:
    for name, expected in EXPECTED_PRIVATE_HASHES.items():
        actual = sha256_file(METADATA_ROOT / name)
        if actual != expected:
            raise ValueError(f"private metadata digest mismatch: {name}: {actual}")


def load_audio(path: Path) -> torch.Tensor:
    values, sample_rate = sf.read(path, dtype="float32", always_2d=False)
    if sample_rate != 16_000 or values.ndim != 1 or values.size == 0:
        raise ValueError("audio must be non-empty mono 16 kHz")
    if values.size > int(MAX_AUDIO_SECONDS * 16_000):
        raise ValueError("audio exceeds configured maximum duration")
    if not np.isfinite(values).all():
        raise ValueError("audio contains non-finite values")
    return torch.from_numpy(np.ascontiguousarray(values))


def ctc_collapse(values: Sequence[int], blank: int = 0) -> tuple[int, ...]:
    output: list[int] = []
    previous: int | None = None
    for value in values:
        if value != blank and value != previous:
            output.append(value)
        previous = value
    return tuple(output)


def edit_breakdown(reference: Sequence[int], hypothesis: Sequence[int]) -> tuple[int, int, int]:
    # Dynamic programming values are (cost, substitutions, deletions, insertions).
    previous = [(j, 0, 0, j) for j in range(len(hypothesis) + 1)]
    for i, ref in enumerate(reference, start=1):
        current = [(i, 0, i, 0)]
        for j, hyp in enumerate(hypothesis, start=1):
            if ref == hyp:
                diagonal = previous[j - 1]
            else:
                base = previous[j - 1]
                diagonal = (base[0] + 1, base[1] + 1, base[2], base[3])
            deletion_base = previous[j]
            deletion = (deletion_base[0] + 1, deletion_base[1], deletion_base[2] + 1, deletion_base[3])
            insertion_base = current[j - 1]
            insertion = (insertion_base[0] + 1, insertion_base[1], insertion_base[2], insertion_base[3] + 1)
            current.append(min(diagonal, deletion, insertion))
        previous = current
    _, substitutions, deletions, insertions = previous[-1]
    return substitutions, deletions, insertions


def artifact_binding(*, inventory_sha256: str, teacher_sha256: str, corpus_sha256: str) -> dict[str, Any]:
    return {
        "schema_version": "1.0",
        "run_mode": RUN_MODE,
        "seed": SEED,
        "archive_sha256": ARCHIVE_SHA256,
        "corpus_index_sha256": corpus_sha256,
        "inventory_sha256": inventory_sha256,
        "teacher_model_id": TEACHER_MODEL_ID,
        "teacher_revision": TEACHER_REVISION,
        "teacher_checkpoint_sha256": teacher_sha256,
        "evidence_scope": "adult_tamil_engineering_proxy",
        "production_ready": False,
    }


set_seed(SEED)
verify_metadata()
DEVICE = torch.device("cuda")
GPU_TOTAL_BYTES = torch.cuda.get_device_properties(0).total_memory
USE_BF16 = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print({"gpu_bytes": GPU_TOTAL_BYTES, "precision": str(AMP_DTYPE), "metadata_verified": True})

In [ ]:
from google.colab import userdata
from huggingface_hub import login, snapshot_download

hf_token = userdata.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    print("HF_TOKEN was not found in Colab Secrets. Complete the interactive Hugging Face login.")
    login(add_to_git_credential=False)
del hf_token

snapshot = Path(
    snapshot_download(
        repo_id=TEACHER_MODEL_ID,
        revision=TEACHER_REVISION,
        allow_patterns=["*.nemo"],
        token=True,
    )
)
nemo_files = tuple(snapshot.glob("*.nemo"))
if len(nemo_files) != 1:
    raise ValueError(f"expected exactly one .nemo file at the pinned revision, found {len(nemo_files)}")
teacher_nemo = nemo_files[0]
teacher_nemo_sha256 = sha256_file(teacher_nemo)

from nemo.collections.asr.models.hybrid_rnnt_ctc_bpe_models import (
    EncDecHybridRNNTCTCBPEModel,
)

teacher = EncDecHybridRNNTCTCBPEModel.restore_from(
    restore_path=str(teacher_nemo), map_location=DEVICE
).to(DEVICE)
teacher.freeze()
teacher.eval()
if not hasattr(teacher, "encoder") or not hasattr(teacher, "preprocessor"):
    raise TypeError(f"unexpected teacher contract: {type(teacher).__name__}")
encoder_layers = getattr(teacher.encoder, "layers", None)
encoder_hidden = int(getattr(teacher.encoder, "_feat_out", 0))
if encoder_layers is None or len(encoder_layers) != 17 or encoder_hidden != 512:
    raise ValueError(
        f"teacher probe mismatch: class={type(teacher).__name__}, "
        f"blocks={None if encoder_layers is None else len(encoder_layers)}, hidden={encoder_hidden}"
    )

with torch.inference_mode():
    probe_audio = torch.zeros((1, 16_000), dtype=torch.float32, device=DEVICE)
    probe_lengths = torch.tensor([16_000], dtype=torch.long, device=DEVICE)
    probe_encoded, probe_frame_lengths = teacher(
        input_signal=probe_audio, input_signal_length=probe_lengths
    )
if probe_encoded.ndim != 3 or probe_encoded.shape[1] != 512:
    raise ValueError(f"unexpected teacher encoder shape: {tuple(probe_encoded.shape)}")

teacher_probe = {
    "schema_version": "1.0",
    "model_id": TEACHER_MODEL_ID,
    "revision": TEACHER_REVISION,
    "nemo_revision": AI4BHARAT_NEMO_REVISION,
    "nemo_file_sha256": teacher_nemo_sha256,
    "model_class": type(teacher).__name__,
    "encoder_class": type(teacher.encoder).__name__,
    "encoder_blocks": len(encoder_layers),
    "hidden_dimension": encoder_hidden,
    "probe_shape_bdt": list(probe_encoded.shape),
    "probe_frame_lengths": probe_frame_lengths.detach().cpu().tolist(),
    "direct_phoneme_posteriors_supported": False,
}
write_json_atomic(DRIVE_OUTPUTS / "teacher-probe.json", teacher_probe)
del probe_audio, probe_encoded, probe_frame_lengths
torch.cuda.empty_cache()
print(teacher_probe)

In [ ]:
if not ARCHIVE_PATH.is_file():
    drive_archive = DRIVE_INPUTS / ARCHIVE_PATH.name
    if drive_archive.is_file():
        shutil.copy2(drive_archive, ARCHIVE_PATH)
    elif DOWNLOAD_CORPUS_IF_MISSING:
        run(
            [
                "curl",
                "--location",
                "--fail",
                "--retry",
                "20",
                "--retry-all-errors",
                "--continue-at",
                "-",
                "--output",
                str(ARCHIVE_PATH),
                ARCHIVE_URL,
            ]
        )
    else:
        raise FileNotFoundError(f"Upload the corpus archive to {drive_archive}")

if ARCHIVE_PATH.stat().st_size != ARCHIVE_SIZE:
    raise ValueError(f"archive size mismatch: {ARCHIVE_PATH.stat().st_size} != {ARCHIVE_SIZE}")
if sha256_file(ARCHIVE_PATH) != ARCHIVE_SHA256:
    raise ValueError("archive SHA-256 mismatch")
extraction_state = safe_extract_tar(ARCHIVE_PATH, EXTRACT_ROOT)

wav_count = sum(1 for _ in EXTRACT_ROOT.rglob("*.wav"))
txt_count = sum(1 for _ in EXTRACT_ROOT.rglob("*.txt"))
if (wav_count, txt_count) != (89_401, 89_401):
    raise ValueError(f"extracted pair count mismatch: wav={wav_count}, txt={txt_count}")

corpus_evidence = json.loads(
    (METADATA_ROOT / "openslr127-corpus-evidence.json").read_text(encoding="utf-8")
)
frozen_index = json.loads(
    (METADATA_ROOT / "openslr127-frozen-index.json").read_text(encoding="utf-8")
)
if corpus_evidence["accepted_count"] != 89_401 or corpus_evidence["rejections"] != {}:
    raise ValueError("corpus evidence does not prove complete zero-rejection validation")
if frozen_index["record_count"] != 89_387 or frozen_index["speaker_count"] != 638:
    raise ValueError("frozen corpus index differs from the verified training population")
corpus_index_sha256 = frozen_index["corpus_index_sha256"]
print(
    {
        "archive_sha256": ARCHIVE_SHA256,
        "extraction": extraction_state,
        "wav_count": wav_count,
        "transcript_count": txt_count,
        "training_eligible": frozen_index["record_count"],
    }
)

In [ ]:
TAMIL_WORD = re.compile(r"[\u0b80-\u0bff]+")
candidate_path = METADATA_ROOT / "tamil-phoneme-candidate.json"
candidate = json.loads(candidate_path.read_text(encoding="utf-8"))
approved_path = DRIVE_INPUTS / "tamil-phoneme-contract.expert-reviewed.json"

if approved_path.is_file():
    inventory = json.loads(approved_path.read_text(encoding="utf-8"))
    if inventory.get("expert_approved") is not True or inventory.get("production_ready") is not False:
        raise ValueError("reviewed inventory flags are invalid")
    reviewer = inventory.get("reviewer") or {}
    if reviewer.get("attests_inventory_reviewed") is not True:
        raise ValueError("attests_inventory_reviewed must be true")
    tokens = inventory["tokens"]
    inventory_status = "expert_reviewed_inventory_not_clinically_validated"
    inventory_sha256 = sha256_file(approved_path)
else:
    if not ALLOW_PROVISIONAL_RESEARCH_RUN:
        raise RuntimeError(
            "No expert-reviewed inventory supplied. Upload it or explicitly enable the provisional research run."
        )
    if candidate.get("expert_approved") is not False or candidate.get("production_ready") is not False:
        raise ValueError("candidate inventory has unsafe approval flags")
    tokens = candidate["tokens"]
    inventory_status = "provisional_research_only_retrain_after_clinician_review"
    inventory_sha256 = sha256_file(candidate_path)

if tokens[0] != "<blank>" or len(tokens) != len(set(tokens)):
    raise ValueError("inventory must have a unique index-zero CTC blank")
token_to_id = {token: index for index, token in enumerate(tokens)}
lexicon_payload = json.loads((METADATA_ROOT / "full-lexicon.json").read_text(encoding="utf-8"))
lexicon = {entry["word"]: tuple(entry["units"]) for entry in lexicon_payload["entries"]}
assignments = json.loads(
    (METADATA_ROOT / "speaker-assignments.json").read_text(encoding="utf-8")
)
records_path = METADATA_ROOT / "records.jsonl"
targets_cache = DRIVE_PRIVATE / f"training-items-{inventory_sha256}.jsonl"
targets_summary_path = DRIVE_PRIVATE / f"training-items-{inventory_sha256}.summary.json"

if FORCE_REBUILD_TARGETS or not (targets_cache.is_file() and targets_summary_path.is_file()):
    temporary = targets_cache.with_suffix(".jsonl.tmp")
    split_counts: Counter[str] = Counter()
    unscorable = Counter()
    with records_path.open(encoding="utf-8") as source, temporary.open(
        "w", encoding="utf-8", newline="\n"
    ) as output:
        for line in tqdm(source, total=89_387, desc="Phoneme targets"):
            record = json.loads(line)
            transcript_path = (EXTRACT_ROOT / record["transcript_rel_path"]).resolve()
            transcript_path.relative_to(EXTRACT_ROOT.resolve())
            normalized = unicodedata.normalize("NFC", transcript_path.read_text(encoding="utf-8")).strip()
            if hashlib.sha256(normalized.encode("utf-8")).hexdigest() != record["transcript_sha256"]:
                raise ValueError("transcript digest mismatch while generating targets")
            units: list[str] = []
            missing = False
            for word in TAMIL_WORD.findall(normalized):
                word_units = lexicon.get(word)
                if word_units is None or any(unit not in token_to_id for unit in word_units):
                    missing = True
                    break
                units.extend(word_units)
            if missing or not units:
                unscorable["unknown_or_unreviewed_phone"] += 1
                continue
            split = assignments[record["speaker_id"]]
            row = {
                "audio_rel_path": record["audio_rel_path"],
                "audio_sha256": record["audio_sha256"],
                "split": split,
                "target_ids": [token_to_id[unit] for unit in units],
            }
            output.write(json.dumps(row, separators=(",", ":")) + "\n")
            split_counts[split] += 1
    os.replace(temporary, targets_cache)
    targets_summary = {
        "schema_version": "1.0",
        "inventory_sha256": inventory_sha256,
        "inventory_status": inventory_status,
        "corpus_index_sha256": corpus_index_sha256,
        "split_counts": dict(split_counts),
        "unscorable": dict(unscorable),
        "private_target_index_sha256": sha256_file(targets_cache),
    }
    write_json_atomic(targets_summary_path, targets_summary)
else:
    targets_summary = json.loads(targets_summary_path.read_text(encoding="utf-8"))
    if targets_summary["corpus_index_sha256"] != corpus_index_sha256:
        raise ValueError("cached targets use a different corpus index")


@dataclass(frozen=True)
class TrainingItem:
    audio_rel_path: str
    audio_sha256: str
    split: str
    target_ids: tuple[int, ...]


items_by_split: dict[str, list[TrainingItem]] = {"train": [], "validation": [], "test": []}
with targets_cache.open(encoding="utf-8") as source:
    for line in source:
        row = json.loads(line)
        split_items = items_by_split[row["split"]]
        limit = SPLIT_LIMITS[row["split"]]
        if limit is None or len(split_items) < limit:
            split_items.append(
                TrainingItem(
                    audio_rel_path=row["audio_rel_path"],
                    audio_sha256=row["audio_sha256"],
                    split=row["split"],
                    target_ids=tuple(row["target_ids"]),
                )
            )

if any(not values for values in items_by_split.values()):
    raise ValueError("every frozen split must contain scorable items")
run_binding = artifact_binding(
    inventory_sha256=inventory_sha256,
    teacher_sha256=teacher_nemo_sha256,
    corpus_sha256=corpus_index_sha256,
)
run_binding["inventory_status"] = inventory_status
run_binding["split_counts_used"] = {name: len(values) for name, values in items_by_split.items()}
run_binding["allow_provisional_research_run"] = ALLOW_PROVISIONAL_RESEARCH_RUN
write_json_atomic(DRIVE_OUTPUTS / "run-manifest.json", run_binding)
print({"inventory_status": inventory_status, "vocabulary_size": len(tokens), **run_binding["split_counts_used"]})

In [ ]:
class NemoAcousticEncoder(nn.Module):
    def __init__(self, preprocessor: nn.Module, encoder: nn.Module, hidden_size: int) -> None:
        super().__init__()
        self.preprocessor = preprocessor
        self.encoder = encoder
        self.hidden_size = hidden_size

    def forward(self, audio: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        processed, processed_lengths = self.preprocessor(input_signal=audio, length=lengths)
        encoded, frame_lengths = self.encoder(audio_signal=processed, length=processed_lengths)
        if encoded.ndim != 3 or encoded.shape[1] != self.hidden_size:
            raise ValueError(f"unexpected NeMo encoded shape: {tuple(encoded.shape)}")
        return encoded.transpose(1, 2), frame_lengths.long()


class ReferencePhonemeCtc(nn.Module):
    def __init__(self, acoustic: NemoAcousticEncoder, vocabulary_size: int) -> None:
        super().__init__()
        self.acoustic = acoustic
        self.phoneme_head = nn.Linear(acoustic.hidden_size, vocabulary_size)

    def forward(self, audio: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        hidden, frame_lengths = self.acoustic(audio, lengths)
        return self.phoneme_head(hidden), hidden, frame_lengths


reference_model = ReferencePhonemeCtc(
    NemoAcousticEncoder(teacher.preprocessor, teacher.encoder, encoder_hidden), len(tokens)
).to(DEVICE)
del teacher
torch.cuda.empty_cache()


def collate_items(batch_items: Sequence[TrainingItem]) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    waveforms = [load_audio(EXTRACT_ROOT / item.audio_rel_path) for item in batch_items]
    targets = [torch.tensor(item.target_ids, dtype=torch.long) for item in batch_items]
    return (
        pad_sequence(waveforms, batch_first=True).to(DEVICE, non_blocking=True),
        torch.tensor([waveform.numel() for waveform in waveforms], dtype=torch.long, device=DEVICE),
        pad_sequence(targets, batch_first=True, padding_value=-1).to(DEVICE),
        torch.tensor([target.numel() for target in targets], dtype=torch.long, device=DEVICE),
    )


def batches(values: Sequence[TrainingItem], batch_size: int, *, seed: int | None = None) -> Iterator[list[TrainingItem]]:
    order = list(range(len(values)))
    if seed is not None:
        random.Random(seed).shuffle(order)
    for start in range(0, len(order), batch_size):
        yield [values[index] for index in order[start : start + batch_size]]


def required_ctc_frames(target: torch.Tensor, length: int) -> int:
    values = target[:length].tolist()
    return len(values) + sum(a == b for a, b in zip(values, values[1:]))


def phoneme_ctc_loss(
    logits: torch.Tensor,
    frame_lengths: torch.Tensor,
    padded_targets: torch.Tensor,
    target_lengths: torch.Tensor,
) -> torch.Tensor:
    flattened = torch.cat(
        [row[: int(length)] for row, length in zip(padded_targets, target_lengths.tolist())]
    )
    return nn.functional.ctc_loss(
        logits.float().log_softmax(dim=-1).transpose(0, 1),
        flattened,
        frame_lengths.long(),
        target_lengths.long(),
        blank=0,
        reduction="mean",
        zero_infinity=False,
    )


@torch.inference_mode()
def evaluate_ctc(model: nn.Module, values: Sequence[TrainingItem], *, description: str) -> dict[str, Any]:
    model.eval()
    substitutions = deletions = insertions = references = exact = scorable = 0
    loss_sum = 0.0
    for batch_items in tqdm(batches(values, BATCH_SIZE), total=math.ceil(len(values) / BATCH_SIZE), desc=description):
        audio, lengths, padded_targets, target_lengths = collate_items(batch_items)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits, _, frame_lengths = model(audio, lengths)
        for index, item in enumerate(batch_items):
            target_length = int(target_lengths[index])
            if required_ctc_frames(padded_targets[index], target_length) > int(frame_lengths[index]):
                continue
            reference = tuple(int(value) for value in padded_targets[index, :target_length].tolist())
            path = logits[index, : int(frame_lengths[index])].argmax(dim=-1).tolist()
            hypothesis = ctc_collapse(path)
            sub, delete, insert = edit_breakdown(reference, hypothesis)
            substitutions += sub
            deletions += delete
            insertions += insert
            references += len(reference)
            exact += int(sub + delete + insert == 0)
            scorable += 1
        compatible = [
            required_ctc_frames(row, int(length)) <= int(frames)
            for row, length, frames in zip(padded_targets, target_lengths, frame_lengths)
        ]
        if all(compatible):
            loss_sum += float(phoneme_ctc_loss(logits, frame_lengths, padded_targets, target_lengths))
    if scorable == 0 or references == 0:
        raise ValueError("evaluation has no CTC-compatible records")
    errors = substitutions + deletions + insertions
    return {
        "record_count": len(values),
        "scorable_count": scorable,
        "unscorable_count": len(values) - scorable,
        "reference_phones": references,
        "substitutions": substitutions,
        "deletions": deletions,
        "insertions": insertions,
        "substitution_rate": substitutions / references,
        "deletion_rate": deletions / references,
        "insertion_rate": insertions / references,
        "phoneme_error_rate": errors / references,
        "exact_sequence_accuracy": exact / scorable,
        "mean_compatible_batch_loss": loss_sum / max(1, math.ceil(len(values) / BATCH_SIZE)),
        "evidence_scope": "adult_tamil_engineering_proxy",
    }


REFERENCE_STAGE_CONFIG = {
    "head_only": {"encoder_lr": 0.0, "head_lr": 1e-3},
    "top_encoder_blocks": {"encoder_lr": 1e-5, "head_lr": 1e-4, "top_blocks": 4},
    "full_encoder": {"encoder_lr": 1e-6, "head_lr": 5e-5},
}


def configure_reference_stage(model: ReferencePhonemeCtc, stage: str) -> list[dict[str, Any]]:
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    for parameter in model.phoneme_head.parameters():
        parameter.requires_grad_(True)
    groups: list[dict[str, Any]] = [
        {"params": list(model.phoneme_head.parameters()), "lr": REFERENCE_STAGE_CONFIG[stage]["head_lr"]}
    ]
    if stage == "top_encoder_blocks":
        layers = model.acoustic.encoder.layers
        if len(layers) != 17:
            raise ValueError("top-block unfreeze requires the probed 17-block teacher")
        for parameter in layers[-4:].parameters():
            parameter.requires_grad_(True)
        groups.append({"params": list(layers[-4:].parameters()), "lr": 1e-5})
    elif stage == "full_encoder":
        for parameter in model.acoustic.encoder.parameters():
            parameter.requires_grad_(True)
        groups.append({"params": list(model.acoustic.encoder.parameters()), "lr": 1e-6})
    elif stage != "head_only":
        raise ValueError(f"unknown reference stage: {stage}")
    return groups


def atomic_torch_save(payload: object, path: Path) -> None:
    local = WORK_ROOT / f"{path.name}.{os.getpid()}.tmp"
    torch.save(payload, local)
    path.parent.mkdir(parents=True, exist_ok=True)
    remote_tmp = path.with_suffix(path.suffix + ".tmp")
    shutil.copy2(local, remote_tmp)
    os.replace(remote_tmp, path)
    local.unlink()


def load_bound_checkpoint(path: Path, model: nn.Module, optimizer: torch.optim.Optimizer | None = None) -> dict[str, Any]:
    state = torch.load(path, map_location="cpu")
    if state["binding"] != run_binding:
        raise ValueError(f"checkpoint binding mismatch: {path}")
    model.load_state_dict(state["model_state"])
    if optimizer is not None and state.get("optimizer_state") is not None:
        optimizer.load_state_dict(state["optimizer_state"])
    return state


def train_reference_stage(
    model: ReferencePhonemeCtc,
    stage: str,
    epochs: int,
    *,
    initialize_from: Path | None,
) -> tuple[Path, dict[str, Any]]:
    stage_dir = DRIVE_PRIVATE / "reference" / stage
    stage_dir.mkdir(parents=True, exist_ok=True)
    last_path = stage_dir / "last.pt"
    groups = configure_reference_stage(model, stage)
    optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler(enabled=not USE_BF16)
    start_epoch = 0
    history: list[dict[str, Any]] = []
    if last_path.is_file():
        state = load_bound_checkpoint(last_path, model, optimizer)
        scaler.load_state_dict(state["scaler_state"])
        start_epoch = state["epoch"] + 1
        history = state["history"]
        print(f"Resuming {stage} at epoch {start_epoch}")
    elif initialize_from is not None:
        load_bound_checkpoint(initialize_from, model)

    for epoch in range(start_epoch, epochs):
        model.train()
        if stage == "head_only":
            model.acoustic.eval()
        optimizer.zero_grad(set_to_none=True)
        running_loss = 0.0
        optimizer_steps = 0
        skipped = 0
        train_batches = list(batches(items_by_split["train"], BATCH_SIZE, seed=SEED + epoch))
        for batch_index, batch_items in enumerate(tqdm(train_batches, desc=f"{stage} epoch {epoch + 1}")):
            audio, lengths, padded_targets, target_lengths = collate_items(batch_items)
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                logits, _, frame_lengths = model(audio, lengths)
                if any(
                    required_ctc_frames(row, int(length)) > int(frames)
                    for row, length, frames in zip(padded_targets, target_lengths, frame_lengths)
                ):
                    skipped += len(batch_items)
                    continue
                loss = phoneme_ctc_loss(logits, frame_lengths, padded_targets, target_lengths)
                loss = loss / GRADIENT_ACCUMULATION
            if not torch.isfinite(loss):
                raise FloatingPointError("non-finite reference CTC loss")
            scaler.scale(loss).backward()
            running_loss += float(loss.detach()) * GRADIENT_ACCUMULATION
            if (batch_index + 1) % GRADIENT_ACCUMULATION == 0 or batch_index + 1 == len(train_batches):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                optimizer_steps += 1
        validation = evaluate_ctc(model, items_by_split["validation"], description=f"{stage} validation")
        epoch_result = {
            "epoch": epoch,
            "training_loss": running_loss / max(1, len(train_batches) - skipped),
            "optimizer_steps": optimizer_steps,
            "skipped_ctc_incompatible": skipped,
            "validation": validation,
        }
        history.append(epoch_result)
        checkpoint = {
            "binding": run_binding,
            "stage": stage,
            "epoch": epoch,
            "history": history,
            "model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},
            "optimizer_state": optimizer.state_dict(),
            "scaler_state": scaler.state_dict(),
        }
        epoch_path = stage_dir / f"epoch-{epoch + 1:03d}-per-{validation['phoneme_error_rate']:.6f}.pt"
        atomic_torch_save(checkpoint, epoch_path)
        atomic_torch_save(checkpoint, last_path)
        ranked = sorted(
            stage_dir.glob("epoch-*.pt"),
            key=lambda path: float(path.stem.rsplit("-per-", 1)[1]),
        )
        for old in ranked[3:]:
            old.unlink()
        print(epoch_result)

    candidates = sorted(
        stage_dir.glob("epoch-*.pt"),
        key=lambda path: float(path.stem.rsplit("-per-", 1)[1]),
    )
    if not candidates:
        raise RuntimeError(f"no checkpoints produced for {stage}")
    best_path = candidates[0]
    best_state = load_bound_checkpoint(best_path, model)
    return best_path, min(best_state["history"], key=lambda row: row["validation"]["phoneme_error_rate"])


reference_stages: dict[str, dict[str, Any]] = {}
head_path, head_result = train_reference_stage(
    reference_model, "head_only", REFERENCE_EPOCHS["head_only"], initialize_from=None
)
reference_stages["head_only"] = {"checkpoint": str(head_path), **head_result}
top_path, top_result = train_reference_stage(
    reference_model,
    "top_encoder_blocks",
    REFERENCE_EPOCHS["top_encoder_blocks"],
    initialize_from=head_path,
)
reference_stages["top_encoder_blocks"] = {"checkpoint": str(top_path), **top_result}

top_improvement = (
    head_result["validation"]["phoneme_error_rate"]
    - top_result["validation"]["phoneme_error_rate"]
)
selected_path = top_path
if top_improvement >= 0.005:
    full_path, full_result = train_reference_stage(
        reference_model,
        "full_encoder",
        REFERENCE_EPOCHS["full_encoder"],
        initialize_from=top_path,
    )
    reference_stages["full_encoder"] = {"checkpoint": str(full_path), **full_result}
    selected_path = min(
        (top_path, full_path),
        key=lambda path: float(path.stem.rsplit("-per-", 1)[1]),
    )
else:
    reference_stages["full_encoder"] = {
        "status": "skipped",
        "reason": "top-stage validation PER improvement below 0.005",
        "measured_improvement": top_improvement,
    }

selected_state = load_bound_checkpoint(selected_path, reference_model)
reference_validation = evaluate_ctc(
    reference_model, items_by_split["validation"], description="selected reference validation"
)
reference_test = evaluate_ctc(reference_model, items_by_split["test"], description="selected reference test")
reference_checkpoint_sha256 = sha256_file(selected_path)
reference_metrics = {
    "schema_version": "1.0",
    "binding": run_binding,
    "stages": reference_stages,
    "selected_checkpoint_sha256": reference_checkpoint_sha256,
    "selected_stage": selected_state["stage"],
    "validation": reference_validation,
    "test": reference_test,
    "production_ready": False,
}
write_json_atomic(DRIVE_OUTPUTS / "reference-metrics.json", reference_metrics)
print(reference_metrics)

In [ ]:
FEATURE_BINDING = {
    **run_binding,
    "reference_checkpoint_sha256": reference_checkpoint_sha256,
    "feature_layer": "selected_reference_encoder_output",
    "dtype": "float16",
}
feature_root = DRIVE_PRIVATE / "teacher-features"
feature_root.mkdir(parents=True, exist_ok=True)


@torch.inference_mode()
def extract_feature_shards(
    model: ReferencePhonemeCtc, split: str, values: Sequence[TrainingItem]
) -> list[dict[str, Any]]:
    model.eval()
    shard_records: list[dict[str, Any]] = []
    chunks = [values[start : start + FEATURE_SHARD_SIZE] for start in range(0, len(values), FEATURE_SHARD_SIZE)]
    for shard_index, chunk in enumerate(tqdm(chunks, desc=f"teacher features: {split}")):
        shard_path = feature_root / f"{split}-{shard_index:05d}.pt"
        if shard_path.is_file():
            existing = torch.load(shard_path, map_location="cpu")
            if existing.get("binding") != FEATURE_BINDING or len(existing.get("items", [])) != len(chunk):
                raise ValueError(f"existing feature shard binding mismatch: {shard_path}")
        else:
            feature_items: list[dict[str, Any]] = []
            for item in chunk:
                audio = load_audio(EXTRACT_ROOT / item.audio_rel_path).unsqueeze(0).to(DEVICE)
                lengths = torch.tensor([audio.shape[1]], dtype=torch.long, device=DEVICE)
                with torch.autocast("cuda", dtype=AMP_DTYPE):
                    _, hidden, frame_lengths = model(audio, lengths)
                frames = int(frame_lengths[0])
                feature_items.append(
                    {
                        "audio_rel_path": item.audio_rel_path,
                        "audio_sha256": item.audio_sha256,
                        "target_ids": item.target_ids,
                        "teacher_hidden": hidden[0, :frames].detach().to(dtype=torch.float16, device="cpu"),
                    }
                )
            atomic_torch_save(
                {"schema_version": "1.0", "binding": FEATURE_BINDING, "split": split, "items": feature_items},
                shard_path,
            )
        shard_records.append(
            {"name": shard_path.name, "record_count": len(chunk), "sha256": sha256_file(shard_path)}
        )
    return shard_records


feature_splits = {
    split: extract_feature_shards(reference_model, split, values)
    for split, values in items_by_split.items()
}
teacher_feature_manifest = {
    "schema_version": "1.0",
    "binding": FEATURE_BINDING,
    "feature_dimension": encoder_hidden,
    "splits": {
        split: {
            "record_count": sum(shard["record_count"] for shard in shards),
            "shard_count": len(shards),
            "shards": shards,
        }
        for split, shards in feature_splits.items()
    },
    "contains_audio": False,
    "contains_full_transcripts": False,
    "storage_policy": "private_google_drive_only",
    "production_ready": False,
}
write_json_atomic(DRIVE_OUTPUTS / "teacher-feature-manifest.json", teacher_feature_manifest)
teacher_feature_manifest_sha256 = sha256_file(DRIVE_OUTPUTS / "teacher-feature-manifest.json")
print(
    {
        "teacher_feature_manifest_sha256": teacher_feature_manifest_sha256,
        "split_records": {
            split: value["record_count"] for split, value in teacher_feature_manifest["splits"].items()
        },
    }
)

In [ ]:
reference_model.to("cpu")
del reference_model
torch.cuda.empty_cache()


@dataclass(frozen=True)
class StudentOutput:
    logits: torch.Tensor
    hidden: torch.Tensor
    frame_lengths: torch.Tensor


class WaveFrontend(nn.Module):
    def __init__(self, channels: int = 48) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv1d(1, channels, kernel_size=400, stride=160),
            nn.GELU(),
            nn.Conv1d(channels, channels, kernel_size=5, stride=2, padding=2),
            nn.GELU(),
        )

    @staticmethod
    def output_lengths(lengths: torch.Tensor) -> torch.Tensor:
        first = torch.div(lengths - 400, 160, rounding_mode="floor") + 1
        second = torch.div(first + 1, 2, rounding_mode="floor")
        return second.clamp_min(0).long()

    def forward(self, audio: torch.Tensor) -> torch.Tensor:
        return self.layers(audio.unsqueeze(1)).transpose(1, 2)


def length_mask(lengths: torch.Tensor, frames: int) -> torch.Tensor:
    return torch.arange(frames, device=lengths.device).unsqueeze(0) < lengths.unsqueeze(1)


class CompactConformerCtc(nn.Module):
    architecture = "compact_conformer"

    def __init__(self, vocabulary_size: int) -> None:
        super().__init__()
        self.frontend = WaveFrontend(48)
        self.input_projection = nn.Linear(48, 128)
        layer = nn.TransformerEncoderLayer(
            d_model=128,
            nhead=4,
            dim_feedforward=512,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=4, enable_nested_tensor=False)
        self.ctc_head = nn.Linear(128, vocabulary_size)
        self.hidden_size = 128

    def forward(self, audio: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        frame_lengths = self.frontend.output_lengths(lengths)
        features = self.input_projection(self.frontend(audio))
        mask = length_mask(frame_lengths, features.shape[1])
        hidden = self.encoder(features, src_key_padding_mask=~mask)
        hidden = hidden.masked_fill(~mask.unsqueeze(-1), 0.0)
        return self.ctc_head(hidden), hidden, frame_lengths


class CompactConvBiGruCtc(nn.Module):
    architecture = "conv_bigru"

    def __init__(self, vocabulary_size: int) -> None:
        super().__init__()
        self.frontend = WaveFrontend(48)
        self.encoder = nn.GRU(
            input_size=48,
            hidden_size=96,
            num_layers=2,
            bidirectional=True,
            dropout=0.1,
            batch_first=True,
        )
        self.ctc_head = nn.Linear(192, vocabulary_size)
        self.hidden_size = 192

    def forward(self, audio: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        frame_lengths = self.frontend.output_lengths(lengths)
        hidden, _ = self.encoder(self.frontend(audio))
        mask = length_mask(frame_lengths, hidden.shape[1])
        hidden = hidden.masked_fill(~mask.unsqueeze(-1), 0.0)
        return self.ctc_head(hidden), hidden, frame_lengths


class DistillationCriterion(nn.Module):
    def __init__(self, teacher_dimension: int, student_dimension: int) -> None:
        super().__init__()
        self.teacher_projection = nn.Linear(teacher_dimension, student_dimension, bias=False)

    def forward(
        self,
        *,
        logits: torch.Tensor,
        student_hidden: torch.Tensor,
        teacher_hidden: torch.Tensor,
        frame_lengths: torch.Tensor,
        padded_targets: torch.Tensor,
        target_lengths: torch.Tensor,
    ) -> dict[str, torch.Tensor]:
        mask = length_mask(frame_lengths, student_hidden.shape[1])
        projected = self.teacher_projection(teacher_hidden)
        supervised = phoneme_ctc_loss(logits, frame_lengths, padded_targets, target_lengths)
        representation = nn.functional.smooth_l1_loss(student_hidden[mask], projected[mask])
        student_normalized = nn.functional.normalize(student_hidden, dim=-1)
        teacher_normalized = nn.functional.normalize(projected, dim=-1)
        pair_mask = mask.unsqueeze(1) & mask.unsqueeze(2)
        relational = nn.functional.mse_loss(
            (student_normalized @ student_normalized.transpose(1, 2))[pair_mask],
            (teacher_normalized @ teacher_normalized.transpose(1, 2))[pair_mask],
        )
        weights = mask.unsqueeze(-1).to(student_hidden.dtype)
        denominator = weights.sum(dim=1).clamp_min(1.0)
        pooled_student = (student_hidden * weights).sum(dim=1) / denominator
        pooled_teacher = (projected * weights).sum(dim=1) / denominator
        sequence = (1.0 - nn.functional.cosine_similarity(pooled_student, pooled_teacher)).mean()
        total = supervised + 0.5 * representation + 0.2 * relational + 0.1 * sequence
        return {
            "total": total,
            "supervised_phoneme_ctc": supervised,
            "masked_smooth_l1": representation,
            "relational_frame_similarity": relational,
            "sequence_consistency": sequence,
        }


def align_teacher_hidden(
    teacher_values: Sequence[torch.Tensor], frame_lengths: torch.Tensor, maximum_frames: int
) -> torch.Tensor:
    aligned: list[torch.Tensor] = []
    for teacher_hidden, raw_length in zip(teacher_values, frame_lengths.tolist()):
        length = int(raw_length)
        resized = nn.functional.interpolate(
            teacher_hidden.float().transpose(0, 1).unsqueeze(0),
            size=length,
            mode="linear",
            align_corners=False,
        ).squeeze(0).transpose(0, 1)
        if length < maximum_frames:
            resized = nn.functional.pad(resized, (0, 0, 0, maximum_frames - length))
        aligned.append(resized)
    return torch.stack(aligned).to(DEVICE)


def feature_batches(entries: Sequence[dict[str, Any]], batch_size: int, seed: int) -> Iterator[list[dict[str, Any]]]:
    order = list(range(len(entries)))
    random.Random(seed).shuffle(order)
    for start in range(0, len(order), batch_size):
        yield [entries[index] for index in order[start : start + batch_size]]


def collate_feature_entries(
    entries: Sequence[dict[str, Any]],
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, list[torch.Tensor]]:
    waveforms = [load_audio(EXTRACT_ROOT / entry["audio_rel_path"]) for entry in entries]
    targets = [torch.tensor(entry["target_ids"], dtype=torch.long) for entry in entries]
    teacher_values = [entry["teacher_hidden"] for entry in entries]
    return (
        pad_sequence(waveforms, batch_first=True).to(DEVICE),
        torch.tensor([waveform.numel() for waveform in waveforms], dtype=torch.long, device=DEVICE),
        pad_sequence(targets, batch_first=True, padding_value=-1).to(DEVICE),
        torch.tensor([target.numel() for target in targets], dtype=torch.long, device=DEVICE),
        teacher_values,
    )


def student_binding(architecture: str) -> dict[str, Any]:
    return {
        **run_binding,
        "reference_checkpoint_sha256": reference_checkpoint_sha256,
        "teacher_feature_manifest_sha256": teacher_feature_manifest_sha256,
        "architecture": architecture,
        "distillation_weights": {
            "supervised_phoneme_ctc": 1.0,
            "masked_smooth_l1": 0.5,
            "relational_frame_similarity": 0.2,
            "sequence_consistency": 0.1,
            "text_posterior_kl": 0.0,
        },
    }


def load_student_checkpoint(
    path: Path,
    model: nn.Module,
    criterion: nn.Module,
    binding: dict[str, Any],
    optimizer: torch.optim.Optimizer | None = None,
) -> dict[str, Any]:
    state = torch.load(path, map_location="cpu")
    if state["binding"] != binding:
        raise ValueError("student checkpoint binding mismatch")
    model.load_state_dict(state["model_state"])
    criterion.load_state_dict(state["criterion_state"])
    if optimizer is not None:
        optimizer.load_state_dict(state["optimizer_state"])
    return state


def train_student(
    model: nn.Module,
    *,
    architecture: str,
    seed: int,
) -> tuple[nn.Module, Path, dict[str, Any]]:
    model = model.to(DEVICE)
    criterion = DistillationCriterion(encoder_hidden, model.hidden_size).to(DEVICE)
    binding = student_binding(architecture)
    output_dir = DRIVE_PRIVATE / "students" / architecture
    output_dir.mkdir(parents=True, exist_ok=True)
    last_path = output_dir / "last.pt"
    optimizer = torch.optim.AdamW(
        [*model.parameters(), *criterion.parameters()], lr=3e-4, weight_decay=0.01
    )
    scaler = torch.cuda.amp.GradScaler(enabled=not USE_BF16)
    start_epoch = 0
    history: list[dict[str, Any]] = []
    if last_path.is_file():
        state = load_student_checkpoint(last_path, model, criterion, binding, optimizer)
        scaler.load_state_dict(state["scaler_state"])
        start_epoch = state["epoch"] + 1
        history = state["history"]
        print(f"Resuming {architecture} at epoch {start_epoch}")

    train_shards = [feature_root / item["name"] for item in feature_splits["train"]]
    for epoch in range(start_epoch, STUDENT_EPOCHS):
        model.train()
        criterion.train()
        shard_order = list(train_shards)
        random.Random(seed + epoch).shuffle(shard_order)
        running = Counter()
        batch_count = 0
        for shard_index, shard_path in enumerate(tqdm(shard_order, desc=f"{architecture} epoch {epoch + 1}")):
            shard = torch.load(shard_path, map_location="cpu")
            if shard["binding"] != FEATURE_BINDING:
                raise ValueError("feature shard binding changed")
            for entries in feature_batches(shard["items"], STUDENT_BATCH_SIZE, seed + epoch + shard_index):
                audio, lengths, padded_targets, target_lengths, teacher_values = collate_feature_entries(entries)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast("cuda", dtype=AMP_DTYPE):
                    logits, hidden, frame_lengths = model(audio, lengths)
                    if any(
                        required_ctc_frames(row, int(length)) > int(frames)
                        for row, length, frames in zip(padded_targets, target_lengths, frame_lengths)
                    ):
                        running["skipped_ctc_incompatible"] += len(entries)
                        continue
                    aligned_teacher = align_teacher_hidden(
                        teacher_values, frame_lengths, hidden.shape[1]
                    ).to(dtype=hidden.dtype)
                    losses = criterion(
                        logits=logits,
                        student_hidden=hidden,
                        teacher_hidden=aligned_teacher,
                        frame_lengths=frame_lengths,
                        padded_targets=padded_targets,
                        target_lengths=target_lengths,
                    )
                if not torch.isfinite(losses["total"]):
                    raise FloatingPointError("non-finite student distillation loss")
                scaler.scale(losses["total"]).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_([*model.parameters(), *criterion.parameters()], 1.0)
                scaler.step(optimizer)
                scaler.update()
                for name, value in losses.items():
                    running[name] += float(value.detach())
                batch_count += 1
        validation = evaluate_ctc(model, items_by_split["validation"], description=f"{architecture} validation")
        epoch_result = {
            "epoch": epoch,
            "losses": {name: value / max(1, batch_count) for name, value in running.items()},
            "validation": validation,
        }
        history.append(epoch_result)
        checkpoint = {
            "binding": binding,
            "epoch": epoch,
            "history": history,
            "model_state": {key: value.detach().cpu() for key, value in model.state_dict().items()},
            "criterion_state": {
                key: value.detach().cpu() for key, value in criterion.state_dict().items()
            },
            "optimizer_state": optimizer.state_dict(),
            "scaler_state": scaler.state_dict(),
        }
        validation_per = validation["phoneme_error_rate"]
        epoch_path = output_dir / f"epoch-{epoch + 1:03d}-per-{validation_per:.6f}.pt"
        atomic_torch_save(checkpoint, epoch_path)
        atomic_torch_save(checkpoint, last_path)
        ranked = sorted(
            output_dir.glob("epoch-*.pt"), key=lambda path: float(path.stem.rsplit("-per-", 1)[1])
        )
        for old in ranked[3:]:
            old.unlink()
        print(epoch_result)

    candidates = sorted(
        output_dir.glob("epoch-*.pt"), key=lambda path: float(path.stem.rsplit("-per-", 1)[1])
    )
    if not candidates:
        raise RuntimeError(f"no checkpoint produced for {architecture}")
    best_path = candidates[0]
    load_student_checkpoint(best_path, model, criterion, binding)
    test_metrics = evaluate_ctc(model, items_by_split["test"], description=f"{architecture} test")
    metrics = {
        "architecture": architecture,
        "checkpoint_sha256": sha256_file(best_path),
        "validation": evaluate_ctc(model, items_by_split["validation"], description=f"{architecture} final validation"),
        "test": test_metrics,
        "parameter_count": sum(parameter.numel() for parameter in model.parameters()),
        "fp32_parameter_bytes": sum(parameter.numel() for parameter in model.parameters()) * 4,
        "production_ready": False,
    }
    return model, best_path, metrics


student_models: dict[str, nn.Module] = {}
student_results: dict[str, dict[str, Any]] = {}
conformer_model, conformer_path, conformer_metrics = train_student(
    CompactConformerCtc(len(tokens)), architecture="compact_conformer", seed=23
)
student_models["compact_conformer"] = conformer_model
student_results["compact_conformer"] = conformer_metrics
conformer_model.to("cpu")
torch.cuda.empty_cache()

bigru_model, bigru_path, bigru_metrics = train_student(
    CompactConvBiGruCtc(len(tokens)), architecture="conv_bigru", seed=29
)
student_models["conv_bigru"] = bigru_model
student_results["conv_bigru"] = bigru_metrics
bigru_model.to("cpu")
torch.cuda.empty_cache()
print(student_results)

In [ ]:
import warnings

from onnxruntime.quantization import QuantType, quantize_dynamic


class ExportStudent(nn.Module):
    def __init__(self, model: nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(self, audio: torch.Tensor, audio_lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        logits, _, frame_lengths = self.model(audio, audio_lengths)
        return logits, frame_lengths


def export_student(architecture: str, model: nn.Module) -> dict[str, Any]:
    model = model.cpu().eval()
    wrapper = ExportStudent(model).eval()
    fp32_path = DRIVE_OUTPUTS / f"{architecture}-phoneme-ctc-fp32.onnx"
    int8_path = DRIVE_OUTPUTS / f"{architecture}-phoneme-ctc-int8.onnx"
    local_fp32 = WORK_ROOT / fp32_path.name
    local_int8 = WORK_ROOT / int8_path.name
    example_audio = torch.zeros((1, 16_000), dtype=torch.float32)
    example_lengths = torch.tensor([16_000], dtype=torch.long)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        torch.onnx.export(
            wrapper,
            (example_audio, example_lengths),
            local_fp32,
            input_names=["audio", "audio_lengths"],
            output_names=["logits", "frame_lengths"],
            dynamic_axes={
                "audio": {0: "batch", 1: "samples"},
                "audio_lengths": {0: "batch"},
                "logits": {0: "batch", 1: "frames"},
                "frame_lengths": {0: "batch"},
            },
            opset_version=17,
            do_constant_folding=True,
        )
    quantize_dynamic(
        model_input=str(local_fp32),
        model_output=str(local_int8),
        weight_type=QuantType.QInt8,
        per_channel=True,
    )
    shutil.copy2(local_fp32, fp32_path)
    shutil.copy2(local_int8, int8_path)
    return {
        "fp32_onnx": fp32_path.name,
        "fp32_sha256": sha256_file(fp32_path),
        "fp32_bytes": fp32_path.stat().st_size,
        "int8_onnx": int8_path.name,
        "int8_sha256": sha256_file(int8_path),
        "int8_bytes": int8_path.stat().st_size,
    }


for architecture, model in student_models.items():
    student_results[architecture]["export"] = export_student(architecture, model)

teacher_test_per = reference_metrics["test"]["phoneme_error_rate"]
for result in student_results.values():
    student_per = result["test"]["phoneme_error_rate"]
    relative = (student_per - teacher_test_per) / teacher_test_per if teacher_test_per > 0 else math.inf
    result["relative_teacher_per_degradation"] = relative
    result["per_gate_passed"] = relative <= 0.10
    result["size_gate_passed"] = result["export"]["int8_bytes"] <= 50_000_000
    result["physical_device_required"] = True
    result["physical_tablet_p95_ms"] = None
    result["physical_device_gate_status"] = "not_measured"

eligible = [
    result
    for result in student_results.values()
    if result["per_gate_passed"] and result["size_gate_passed"]
]
selected = min(
    eligible,
    key=lambda result: (
        result["test"]["phoneme_error_rate"],
        result["export"]["int8_bytes"],
    ),
    default=None,
)
student_comparison = {
    "schema_version": "1.0",
    "binding": run_binding,
    "teacher_feature_manifest_sha256": teacher_feature_manifest_sha256,
    "reference_test": reference_metrics["test"],
    "students": student_results,
    "selection": {
        "maximum_relative_teacher_per_degradation": 0.10,
        "maximum_quantized_bytes": 50_000_000,
        "physical_device_required": True,
        "selected_architecture_for_device_evaluation": (
            None if selected is None else selected["architecture"]
        ),
        "status": (
            "no_student_passed_adult_proxy_gates"
            if selected is None
            else "candidate_only_pending_physical_tablet_and_clinical_gates"
        ),
    },
    "evidence_scope": "adult_tamil_engineering_proxy",
    "inventory_status": inventory_status,
    "production_ready": False,
}
write_json_atomic(DRIVE_OUTPUTS / "student-comparison.json", student_comparison)
print(student_comparison["selection"])

In [ ]:
import platform

result_files = [
    DRIVE_OUTPUTS / "run-manifest.json",
    DRIVE_OUTPUTS / "teacher-probe.json",
    DRIVE_OUTPUTS / "reference-metrics.json",
    DRIVE_OUTPUTS / "teacher-feature-manifest.json",
    DRIVE_OUTPUTS / "student-comparison.json",
    *sorted(DRIVE_OUTPUTS.glob("*-phoneme-ctc-*.onnx")),
]
artifact_hashes = {
    "schema_version": "1.0",
    "created_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "run_mode": RUN_MODE,
    },
    "artifacts": [
        {"name": path.name, "bytes": path.stat().st_size, "sha256": sha256_file(path)}
        for path in result_files
    ],
    "privacy": {
        "contains_audio": False,
        "contains_full_transcripts": False,
        "contains_speaker_or_utterance_identifiers": False,
        "voice_derived_teacher_features_stay_private": True,
    },
    "evidence_scope": "adult_tamil_engineering_proxy",
    "production_ready": False,
}
hashes_path = DRIVE_OUTPUTS / "artifact-hashes.json"
write_json_atomic(hashes_path, artifact_hashes)

bundle_path = DRIVE_OUTPUTS / "vaakmitra-gpu-results.zip"
bundle_tmp = WORK_ROOT / bundle_path.name
with zipfile.ZipFile(bundle_tmp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in [*result_files, hashes_path]:
        archive.write(path, arcname=path.name)
shutil.copy2(bundle_tmp, bundle_path)

print(
    {
        "status": "complete_adult_tamil_engineering_proxy",
        "inventory_status": inventory_status,
        "results_directory": str(DRIVE_OUTPUTS),
        "bundle": str(bundle_path),
        "bundle_sha256": sha256_file(bundle_path),
        "production_ready": False,
    }
)